# Grad Café Admissions Data Pipeline: Amazon S3 to SageMaker

**Course:** Johns Hopkins University — Software Concepts (EN.605.601)  
**Module:** Module 7 — Cloud Computing Assignment (Data Pipeline & Cloud Deployment)  
**Author:** Noella Formin (`Noella96` / `Achaformin@gmail.com`)  
**Workspace:** Amazon SageMaker Notebook Instance (`s3-to-sagemaker-grad-cafe-pipeline`)

---

## Overview
This Jupyter notebook establishes an automated, secure data ingestion pipeline connecting **Amazon S3** object storage to an **Amazon SageMaker** compute environment using the AWS Python SDK (`boto3`). 

### Key Security & Architectural Principles
1. **Zero Hardcoded Secrets:** Credentials are resolved dynamically via the AWS credential provider chain (SageMaker execution role or IAM user environment).
2. **Modular Architecture:** Core boto3 retrieval and verification logic is encapsulated in `s3_fetch.py`.
3. **Idempotent Ingestion:** The dataset is retrieved from the designated S3 bucket and persisted locally as `applicant_data_SM.json`.
4. **Data Integrity Assurance:** Verifies file size, valid JSON format, and schema conformance for downstream analytics.


### Step 1: Environment Setup & Library Imports
Import `boto3`, standard library utilities, and the modular ingestion engine `s3_fetch`.

In [ ]:
import json
import os
import sys

# Ensure module path is accessible
sys.path.insert(0, os.path.abspath("."))

import s3_fetch

print(f"[+] Python executable: {sys.executable}")
print(f"[+] s3_fetch module loaded from: {s3_fetch.__file__}")


### Step 2: Configure S3 Pipeline Parameters
Set bucket and object parameters using environment variables with safe defaults.

In [ ]:
BUCKET_NAME = os.environ.get("S3_BUCKET_NAME", "grad-cafe-nf")
OBJECT_KEY = os.environ.get("S3_OBJECT_KEY", "applicant_data.json")
OUTPUT_PATH = os.environ.get("LOCAL_OUTPUT_PATH", "applicant_data_SM.json")
AWS_REGION = os.environ.get("AWS_REGION", "us-east-1")

print(f"Target S3 Bucket: {BUCKET_NAME}")
print(f"Target S3 Object Key: {OBJECT_KEY}")
print(f"Local Destination: {OUTPUT_PATH}")
print(f"AWS Region: {AWS_REGION}")


### Step 3: Execute S3 Data Transfer via boto3
Call `fetch_grad_cafe_data` to download `applicant_data.json` from S3 and save it as `applicant_data_SM.json`.

In [ ]:
result = s3_fetch.fetch_grad_cafe_data(
    bucket_name=BUCKET_NAME,
    object_key=OBJECT_KEY,
    destination_path=OUTPUT_PATH,
    region_name=AWS_REGION,
)

if result.get("success"):
    print("[SUCCESS] Data successfully ingested from S3 to SageMaker workspace.")
    print(json.dumps(result["metrics"], indent=2))
else:
    print(f"[ERROR] Pipeline execution notice: {result.get('error')}")


### Step 4: Validate Ingested Dataset Schema & Sample Records
Inspect the local `applicant_data_SM.json` file to confirm record count and field structure.

In [ ]:
if os.path.exists(OUTPUT_PATH):
    with open(OUTPUT_PATH, "r", encoding="utf-8") as f:
        data = json.load(f)
    print(f"Total applicants loaded: {len(data):,}")
    print("\nSample Record (First Entry):")
    print(json.dumps(data[0] if data else {}, indent=2))
else:
    print(f"Notice: {OUTPUT_PATH} not found locally. Run ingestion above.")


---
### Step 5: Operational & Cost Management Reminder
> **IMPORTANT:** In accordance with course guidelines and cloud cost management best practices, remember to **STOP** (not delete) your Amazon SageMaker notebook instance and EC2 instance when data processing is complete to prevent unintended AWS charges. Infrastructure will be resumed in Module 8.
